# Muon PG Hit Analysis

Read the 100 GeV muon particle-gun EDM4hep samples and compare 0 degree incidence with 45 degree incidence. Energy is reported in MeV.


## Setup

The notebook uses ROOT `RDataFrame` and the `events` tree, following the lightweight plotting style used in the repository examples.


In [11]:
from __future__ import annotations

from pathlib import Path

import ROOT

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kViridis)
ROOT.EnableImplicitMT()


In [12]:
sample_dir = Path("/home/llr/ilc/shi/data/siwecal_k4sim/output/muon")
samples = {
    "0degree": sample_dir / "mu-_100GeV_0degree.edm4hep.root",
    "45degree": sample_dir / "mu-_100GeV_45deg.edm4hep.root",
}

tree_name = "events"
collection = "SiPadHits"

energy_leaf = f"{collection}.energy"
x_leaf = f"{collection}.position.x"
y_leaf = f"{collection}.position.y"
z_leaf = f"{collection}.position.z"

missing = [path for path in samples.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing input sample(s): " + ", ".join(map(str, missing)))

samples


{'0degree': PosixPath('/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_0degree.edm4hep.root'),
 '45degree': PosixPath('/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_45deg.edm4hep.root')}

## Load Hits

Create one `RDataFrame` per sample, alias the SiPad hit leaves, and convert hit energy from GeV to MeV.


In [13]:
def make_hit_frame(path: Path):
    return (
        ROOT.RDataFrame(tree_name, str(path))
            .Alias("hit_energy", energy_leaf)
            .Alias("hit_x", x_leaf)
            .Alias("hit_y", y_leaf)
            .Alias("hit_z", z_leaf)
            .Define("hit_energy_mev", "1000.0f * hit_energy")
    )

frames = {label: make_hit_frame(path) for label, path in samples.items()}
frames


{'0degree': <cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RLoopManager,void> object at 0x12ed8d00>,
 '45degree': <cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RLoopManager,void> object at 0x13ceb4c0>}

In [14]:
def summarize(label: str, path: Path, df):
    n_events = int(ROOT.RDataFrame(tree_name, str(path)).Count().GetValue())
    n_hits = int(df.Define("hit_count", "hit_energy.size()").Sum("hit_count").GetValue())
    mean_energy_mev = float(df.Mean("hit_energy_mev").GetValue()) if n_hits else 0.0
    total_energy_mev = float(df.Sum("hit_energy_mev").GetValue()) if n_hits else 0.0
    max_energy_mev = float(df.Max("hit_energy_mev").GetValue()) if n_hits else 0.0
    return {
        "sample": label,
        "input_file": str(path),
        "events": n_events,
        "SiPadHits": n_hits,
        "mean_hit_energy_MeV": mean_energy_mev,
        "max_hit_energy_MeV": max_energy_mev,
        "total_hit_energy_MeV": total_energy_mev,
    }

summaries = {
    label: summarize(label, samples[label], frames[label])
    for label in samples
}
summaries


{'0degree': {'sample': '0degree',
  'input_file': '/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_0degree.edm4hep.root',
  'events': 1000,
  'SiPadHits': 18335,
  'mean_hit_energy_MeV': 0.27031479678013026,
  'max_hit_energy_MeV': 36.93924331665039,
  'total_hit_energy_MeV': 4956.221798963688},
 '45degree': {'sample': '45degree',
  'input_file': '/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_45deg.edm4hep.root',
  'events': 1000,
  'SiPadHits': 19881,
  'mean_hit_energy_MeV': 0.35855038510393294,
  'max_hit_energy_MeV': 16.892179489135742,
  'total_hit_energy_MeV': 7128.3402062512905}}

## Common Plot Ranges

Use common axis ranges so the 0 degree and 45 degree hit maps can be compared directly.


In [15]:
def minmax(df, column: str):
    return float(df.Min(column).GetValue()), float(df.Max(column).GetValue())

ranges_by_sample = {
    label: {
        "x": minmax(df, "hit_x"),
        "y": minmax(df, "hit_y"),
        "z": minmax(df, "hit_z"),
        "energy_mev": minmax(df, "hit_energy_mev"),
    }
    for label, df in frames.items()
}

def combine_range(quantity: str, pad: float):
    lows = [values[quantity][0] for values in ranges_by_sample.values()]
    highs = [values[quantity][1] for values in ranges_by_sample.values()]
    low = min(lows)
    high = max(highs)
    if low == high:
        return low - pad, high + pad
    return low - pad, high + pad

x_range = combine_range("x", 5.0)
y_range = combine_range("y", 5.0)
z_range = combine_range("z", 5.0)
energy_range_mev = (0.0, 1.0)

plot_ranges = {
    "x_mm": x_range,
    "y_mm": y_range,
    "z_mm": z_range,
    "energy_MeV": energy_range_mev,
}
plot_ranges


{'x_mm': (-175.49899291992188, 181.00100708007812),
 'y_mm': (-180.99899291992188, 170.00100708007812),
 'z_mm': (-121.875, 120.44999694824219),
 'energy_MeV': (0.0, 1.0)}

## Hit Maps

Each canvas compares the two samples side by side with the same axis limits.


In [16]:
histograms = {}
canvases = {}

canvas_zx = ROOT.TCanvas("canvas_zx_compare", "SiPadHits z-x comparison", 1200, 520)
canvas_zx.Divide(2, 1)

for pad_index, (label, df) in enumerate(frames.items(), start=1):
    hist = df.Histo2D(
        (
            f"h_zx_{label}",
            f"{label}: SiPadHits z-x;z [mm];x [mm];Hits",
            140,
            z_range[0],
            z_range[1],
            120,
            x_range[0],
            x_range[1],
        ),
        "hit_z",
        "hit_x",
    )
    histograms[f"zx_{label}"] = hist
    canvas_zx.cd(pad_index)
    hist.Draw("COLZ")

canvases["zx"] = canvas_zx
canvas_zx.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_zx_compare


In [17]:
canvas_zy = ROOT.TCanvas("canvas_zy_compare", "SiPadHits z-y comparison", 1200, 520)
canvas_zy.Divide(2, 1)

for pad_index, (label, df) in enumerate(frames.items(), start=1):
    hist = df.Histo2D(
        (
            f"h_zy_{label}",
            f"{label}: SiPadHits z-y;z [mm];y [mm];Hits",
            140,
            z_range[0],
            z_range[1],
            120,
            y_range[0],
            y_range[1],
        ),
        "hit_z",
        "hit_y",
    )
    histograms[f"zy_{label}"] = hist
    canvas_zy.cd(pad_index)
    hist.Draw("COLZ")

canvases["zy"] = canvas_zy
canvas_zy.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_zy_compare


## Hit Energy Distribution

The energy spectra are overlaid in MeV with a fixed 0-2 MeV range and 200 bins. The vertical scale uses raw hit counts.


In [18]:
colors = {
    "0degree": ROOT.kBlue + 1,
    "45degree": ROOT.kRed + 1,
}

energy_hists = []
for label, df in frames.items():
    hist = df.Histo1D(
        (
            f"h_energy_mev_{label}",
            "SiPadHits energy;Hit energy [MeV];Hits",
            100,
            energy_range_mev[0],
            energy_range_mev[1],
        ),
        "hit_energy_mev",
    )
    histograms[f"energy_{label}"] = hist
    energy_hists.append((label, hist))

canvas_energy = ROOT.TCanvas("canvas_energy_mev_compare", "SiPadHits energy comparison", 800, 600)
legend = ROOT.TLegend(0.60, 0.72, 0.88, 0.88)

max_count = max(float(hist.GetMaximum()) for _, hist in energy_hists)
for index, (label, hist) in enumerate(energy_hists):
    hist.SetLineColor(colors.get(label, ROOT.kBlack))
    hist.SetLineWidth(2)
    hist.SetMaximum(max_count * 1.15 if max_count > 0 else 1.0)
    hist.Draw("hist" if index == 0 else "hist same")
    legend.AddEntry(hist.GetPtr(), label, "l")

legend.Draw()
canvases["energy"] = canvas_energy
canvas_energy.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_energy_mev_compare


## Energy Peaks

Peak values are the centers of the maximum-count bins in the plotted 0-2 MeV histograms.


In [19]:
energy_peaks = {}
for label, hist in energy_hists:
    max_bin = hist.GetMaximumBin()
    energy_peaks[label] = {
        "peak_energy_MeV": float(hist.GetBinCenter(max_bin)),
        "peak_bin_count": int(hist.GetBinContent(max_bin)),
        "bin_width_MeV": float(hist.GetBinWidth(max_bin)),
    }

energy_peaks


{'0degree': {'peak_energy_MeV': 0.145,
  'peak_bin_count': 2130,
  'bin_width_MeV': 0.01},
 '45degree': {'peak_energy_MeV': 0.20500000000000002,
  'peak_bin_count': 1529,
  'bin_width_MeV': 0.01}}

In [20]:
# Digitized Energy Distribution
# Uses SiPadHitsDigiDigitizedEnergy from the RealDigitizer outputs.
# This shaped slow-sample amplitude is stored in MIP units.
digitized_samples = {
    "0degree digitized": sample_dir / "digitized/mu-_100GeV_0degree_real_digitized.edm4hep.root",
    "45degree digitized": sample_dir / "digitized/mu-_100GeV_45deg_real_digitized.edm4hep.root",
}

missing_digitized = [path for path in digitized_samples.values() if not path.is_file()]
if missing_digitized:
    raise FileNotFoundError("Missing digitized sample(s): " + ", ".join(map(str, missing_digitized)))

digitized_frames = {
    label: ROOT.RDataFrame(tree_name, str(path)).Alias(
        "digitized_energy_mip",
        "SiPadHitsDigiDigitizedEnergy",
    )
    for label, path in digitized_samples.items()
}

digitized_energy_hists = []
for label, df in digitized_frames.items():
    hist = df.Histo1D(
        (
            f"h_digitized_energy_mip_{label.replace(' ', '_')}",
            "RealDigitizer shaped energy;Digitized energy [MIP];Hits",
            200,
            0.0,
            2.0,
        ),
        "digitized_energy_mip",
    )
    digitized_energy_hists.append((label, hist))

canvas_digitized_energy = ROOT.TCanvas(
    "canvas_digitized_energy_mip_compare",
    "RealDigitizer shaped energy comparison",
    800,
    600,
)
legend_digitized = ROOT.TLegend(0.55, 0.72, 0.88, 0.88)

max_digitized_count = max(float(hist.GetMaximum()) for _, hist in digitized_energy_hists)
for index, (label, hist) in enumerate(digitized_energy_hists):
    color = ROOT.kBlue + 1 if label.startswith("0degree") else ROOT.kRed + 1
    hist.SetLineColor(color)
    hist.SetLineWidth(2)
    hist.SetMaximum(max_digitized_count * 1.15 if max_digitized_count > 0 else 1.0)
    hist.Draw("hist" if index == 0 else "hist same")
    legend_digitized.AddEntry(hist.GetPtr(), label, "l")

legend_digitized.Draw()
canvas_digitized_energy.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_digitized_energy_mip_compare


## Notes

- The plotted collection is `SiPadHits`, directly from the ddsim EDM4hep output.
- ROOT stores `SiPadHits.energy` in GeV; this notebook converts it to MeV before reporting and plotting.
- Change the `samples` dictionary in the setup cell to compare other muon PG files.
